# Baseline Model Results

Fits and evaluates the two baseline models (the team-independent historical frequency model and the non-Bayesian team-strength Poisson model).
We expect the Bayesian model to out-perform these two baseline models.

## Imports

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

In [2]:
# Find root directory of project
root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "src" / "config.py").exists())
sys.path.insert(0, str(root))

In [3]:
from src.config import PROCESSED_DATA_DIR, TABLES_DIR, TRAIN_SEASONS, TEST_SEASONS
from src.baseline_models import HistoricalFrequencyBaseline, NonBayesianPoissonBaseline

## Evaluation

### Load Training and Testing Data

In [4]:
df = pd.read_csv(PROCESSED_DATA_DIR / "epl_matches.csv", parse_dates = ["Date"])

train_df = df[df["Season"].isin(TRAIN_SEASONS)].reset_index(drop = True)
test_df = df[df["Season"].isin(TEST_SEASONS)].reset_index(drop = True)

print(f"Train: {len(train_df)} matches")
print(f"Test: {len(test_df)} matches")

Train: 2660 matches
Test: 380 matches


### Historical Frequency Model

In [5]:
freq_model = HistoricalFrequencyBaseline().fit(train_df)
freq_preds = freq_model.predict(test_df)

In [6]:
freq_preds[["HomeTeam", "AwayTeam", "Result", "ProbHomeWin", "ProbDraw", "ProbAwayWin"]].head()

,HomeTeam,AwayTeam,Result,ProbHomeWin,ProbDraw,ProbAwayWin
0,Burnley,Man City,A,0.442481,0.234586,0.322932
1,Arsenal,Nott'm Forest,H,0.442481,0.234586,0.322932
2,Bournemouth,West Ham,D,0.442481,0.234586,0.322932
3,Brighton,Luton,H,0.442481,0.234586,0.322932
4,Everton,Fulham,A,0.442481,0.234586,0.322932


In [7]:
print("Historical Frequency Baseline Model")
print(f"Estimated Home Win Percentage: {freq_model.prob_home_win:.3f}")
print(f"Estimated Draw Percentage: {freq_model.prob_draw:.3f}")
print(f"Estimated Away Win Percentage: {freq_model.prob_away_win:.3f}")

Historical Frequency Baseline Model
Estimated Home Win Percentage: 0.442
Estimated Draw Percentage: 0.235
Estimated Away Win Percentage: 0.323


### Non-Bayesian Poisson Model

In [8]:
non_bayesian_poisson_model = NonBayesianPoissonBaseline().fit(train_df)
non_bayesian_poisson_preds = non_bayesian_poisson_model.predict(test_df)

In [9]:
non_bayesian_poisson_preds[["HomeTeam", "AwayTeam", "Result", "ProbHomeWin", "ProbDraw", "ProbAwayWin"]].head()

,HomeTeam,AwayTeam,Result,ProbHomeWin,ProbDraw,ProbAwayWin
0,Burnley,Man City,A,0.093580,0.181131,0.725289
1,Arsenal,Nott'm Forest,H,0.612166,0.213606,0.174228
2,Bournemouth,West Ham,D,0.336111,0.234080,0.429809
3,Brighton,Luton,H,0.309699,0.273802,0.416498
4,Everton,Fulham,A,0.643576,0.211850,0.144574


In [10]:
for model, preds in [("Historical Frequency", freq_preds), ("Non-Bayesian Poisson", non_bayesian_poisson_preds)]:
    pred_labels = preds[["ProbHomeWin", "ProbDraw", "ProbAwayWin"]].idxmax(axis = 1).map({"ProbHomeWin": "H",
                                                                                          "ProbDraw": "D",
                                                                                          "ProbAwayWin": "A"})
    accuracy = (pred_labels == preds["Result"]).mean()
    print(f"{model} Baseline Model Accuracy: {accuracy:.3f}")

Historical Frequency Baseline Model Accuracy: 0.461
Non-Bayesian Poisson Baseline Model Accuracy: 0.513


### Save Evaluation Results

In [11]:
baseline_model_table_output_directory = (TABLES_DIR / "baseline")
baseline_model_table_output_directory.mkdir(parents = True, exist_ok = True)

freq_preds.to_csv(baseline_model_table_output_directory / "historical_frequency_predictions.csv", index = False)
non_bayesian_poisson_preds.to_csv(baseline_model_table_output_directory / "non_bayesian_poisson_predictions.csv", index = False)